TokaMaker Example: Kink and tearing stability with GPEC
==========
In this example we couple TokaMaker to [GPEC](https://github.com/OpenFUSIONToolkit/GPEC) (the Generalized Perturbed Equilibrium Code) to study how the ideal kink and tearing stability of a DIII-D plasma (shot 147131 at 2300 ms) change as the plasma pressure increases. This is done in several steps:

1) Reconstruct the equilibrium in TokaMaker from a gEQDSK file and measured kinetic profiles.
2) Scale up the pressure, re-solving the equilibrium at each step, and pass each equilibrium to GPEC.
3) Use the $n=1$ ideal energy $\delta W$ computed by GPEC to find the ideal (no-wall) $\beta_N$ limit.
4) Use the tearing stability index $\Delta'$ at the $q=2$ surface, together with the Glasser correction $\Delta_{crit}$, to see how tearing stability changes as the plasma approaches that limit.

**Note:** This example uses external tools, and is not part of the automatically tested example suite. In addition to the usual python packages (see `requirements_GPEC_Coupling.txt`) it requires a working [Julia](https://julialang.org/) installation and a local copy of GPEC. See `README.md` in this folder for setup instructions.

In [ ]:
import os
import sys
import json
import shutil
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize']=(6,6)
plt.rcParams['lines.linewidth']=2
plt.rcParams['lines.markeredgewidth']=2
%matplotlib inline
%config InlineBackend.figure_format = "retina"

### Load TokaMaker library

To load the TokaMaker python module we need to tell python where the module is located. This can be done either through the `PYTHONPATH` environment variable or within a script using `sys.path.append()` as below, where we look for the environment variable `OFT_ROOTPATH` to provide the path to where the OpenFUSIONToolkit is installed (`/Applications/OFT` on macOS).

In [ ]:
tokamaker_python_path = os.getenv('OFT_ROOTPATH')
if tokamaker_python_path is not None:
    sys.path.append(os.path.join(tokamaker_python_path,'python'))
from OpenFUSIONToolkit import OFT_env
from OpenFUSIONToolkit.TokaMaker import TokaMaker
from OpenFUSIONToolkit.TokaMaker.meshing import load_gs_mesh
from OpenFUSIONToolkit.TokaMaker.util import read_eqdsk, get_jphi_from_GS

### Load GPEC

GPEC is written in Julia, so we call it from Python using [juliacall](https://juliapy.github.io/PythonCall.jl/stable/). juliacall is configured through environment variables, which must be set before it is imported:

- `PYTHON_JULIACALL_PROJECT` points to your local copy of GPEC (the folder containing GPEC's `Project.toml`). **Set the `GPEC_PROJECT` environment variable, or replace `/path/to/GPEC` below, with the location of your copy.**
- `PYTHON_JULIACALL_HANDLE_SIGNALS` lets Julia run safely alongside OFT's threads.

The first time GPEC is loaded Julia compiles it, which can take several minutes.

In [ ]:
GPEC_PROJECT = os.getenv('GPEC_PROJECT', '/path/to/gpec/')
if not os.path.isfile(os.path.join(GPEC_PROJECT,'Project.toml')):
    raise FileNotFoundError('Could not find GPEC at "{0}". Set GPEC_PROJECT to your local copy of GPEC.'.format(GPEC_PROJECT))
julia_exe = shutil.which('julia')
if julia_exe is None:
    raise FileNotFoundError('Could not find Julia. Install Julia and make sure "julia" is on your PATH.')

os.environ['PYTHON_JULIACALL_EXE'] = julia_exe
os.environ['PYTHON_JULIACALL_PROJECT'] = GPEC_PROJECT
os.environ['PYTHON_JULIACALL_HANDLE_SIGNALS'] = 'yes'
from juliacall import Main as jl
jl.seval('using GeneralizedPerturbedEquilibrium')
jl.seval('using Logging; global_logger(ConsoleLogger(stderr, Logging.Warn))')  # only show GPEC warnings and errors

### Load input data

We start from a kinetic EFIT reconstruction of the discharge, which consists of:

- `g147131.02300_DIIID_KEFIT`: a gEQDSK file containing the equilibrium.
- `g147131.02300_DIIID_KEFIT_corrected.kin`: the measured ion and electron densities and temperatures, and the $E \times B$ rotation frequency, as functions of the normalized poloidal flux $\hat{\psi}$.

In [ ]:
eqdsk = read_eqdsk('g147131.02300_DIIID_KEFIT')
kinetic = pd.read_csv('g147131.02300_DIIID_KEFIT_corrected.kin', sep=r'\s+')
psi_prof = kinetic['psi'].to_numpy()
kinetic.head()

### Initialize TokaMaker

We now create the OFT runtime environment and a TokaMaker object. Only one OFT environment can exist per python session, so restart the kernel if you need to run this cell again. We then load the DIII-D mesh used in the [DIII-D baseline example](https://github.com/OpenFUSIONToolkit/OpenFUSIONToolkit/blob/main/src/examples/TokaMaker/DIIID/DIIID_baseline_ex.ipynb) and set up the solver, using the vacuum toroidal field ($F_0 = R_0 B_0$) from the gEQDSK file.

In [ ]:
myOFT = OFT_env(nthreads=2)
mygs = TokaMaker(myOFT)

mesh_pts, mesh_lc, mesh_reg, coil_dict, cond_dict = load_gs_mesh('DIIID_mesh.h5')
mygs.setup_mesh(mesh_pts, mesh_lc, mesh_reg)
mygs.setup_regions(cond_dict=cond_dict, coil_dict=coil_dict)
mygs.setup(order=2, F0=abs(eqdsk['rcentr']*eqdsk['bcentr']))

### Reconstruct the equilibrium

Next we set the constraints for the equilibrium reconstruction:

- The plasma current $I_p$ and on-axis pressure from the gEQDSK file.
- The plasma shape, using isoflux points on the gEQDSK's boundary.
- The `F9A` and `F9B` coils as a vertical stability coil pair, which holds the vertically unstable plasma in place.
- Regularization terms that keep the coil currents from growing unnecessarily large.

In [ ]:
mygs.set_coil_vsc({'F9A': 1.0, 'F9B': -1.0})
mygs.set_targets(Ip=abs(eqdsk['ip']), pax=eqdsk['pres'][0])

isoflux_pts = eqdsk['rzout'].copy()
mygs.set_isoflux_constraints(isoflux_pts, weights=50.0*np.ones(isoflux_pts.shape[0]))

coil_names = [
    'ECOILA', 'ECOILB',
    'F1A', 'F2A', 'F3A', 'F4A', 'F5A', 'F6A', 'F7A', 'F8A', 'F9A',
    'F1B', 'F2B', 'F3B', 'F4B', 'F5B', 'F6B', 'F7B', 'F8B', 'F9B',
]
reg_terms = [mygs.coil_reg_term({name: 1.0}, target=0.0, weight=1.0) for name in coil_names]
reg_terms.append(mygs.coil_reg_term({'#VSC': 1.0}, target=0.0, weight=1.0E-2))
mygs.set_coil_reg(reg_terms=reg_terms)

We use the $FF'$ and $P'$ profiles from the gEQDSK file, interpolated onto the $\hat{\psi}$ grid of the kinetic profiles. Before solving for the first time we must initialize the flux with an approximate plasma position and shape: major radius, vertical position, minor radius, elongation and triangularity.

In [ ]:
psi_eqdsk = np.linspace(0.0, 1.0, len(eqdsk['ffprim']))
mygs.set_profiles(
    ffp_prof={'type': 'linterp', 'x': psi_prof, 'y': np.interp(psi_prof, psi_eqdsk, eqdsk['ffprim'])},
    pp_prof={'type': 'linterp', 'x': psi_prof, 'y': np.interp(psi_prof, psi_eqdsk, eqdsk['pprime'])},
)

mygs.init_psi(1.7, 0.0, 0.6, 1.8, 0.4)
mygs.solve()
mygs.print_info()

fig, ax = plt.subplots(1,1)
mygs.plot_machine(fig,ax,coil_colormap='seismic',coil_symmap=True,coil_scale=1.E-6,coil_clabel=r'$I_C$ [MA]')
mygs.plot_psi(fig,ax,vacuum_nlevels=4)
_ = ax.plot(eqdsk['rzout'][:,0], eqdsk['rzout'][:,1],'r--')

## Scanning the plasma pressure

To raise $\beta_N$ we multiply the thermal pressure from the kinetic profiles, $p = e(n_e T_e + n_i T_i)$, by a factor `fraction` and re-solve the equilibrium. Only the pressure should change between scan points, so $I_p$, the plasma shape and the toroidal current density profile $j_\phi(\hat{\psi})$ are held fixed. Note that the thermal pressure is about 75% of the total pressure in the gEQDSK file, so `fraction = 1` gives a lower $\beta_N$ than the reconstruction above.

First we compute $j_\phi$ from the reconstructed equilibrium on the kinetic profile grid. The flux surface averages it requires are not defined exactly at the magnetic axis or the separatrix, so the end points of the grid are moved slightly inward.

In [ ]:
psi_eval = np.clip(psi_prof, 1.E-3, 1.0-1.E-3)
_, f, fp, _, pp = mygs.get_profiles(psi=psi_eval)
_, _, ravgs, _, _, _ = mygs.get_q(psi=psi_eval)
jphi_baseline = get_jphi_from_GS(f*fp, pp, ravgs['<R>'], ravgs['<1/R>'])

p_thermal = 1.602176634E-19*(kinetic['ne(m^-3)']*kinetic['te(eV)'] + kinetic['ni(m^-3)']*kinetic['ti(eV)']).to_numpy()

The function below re-solves the equilibrium for a given `fraction` and returns the resulting $\beta_N$ [%]. The new $P'$ profile is computed from the scaled pressure on the same $\hat{\psi}$ grid, and the fixed $j_\phi$ profile is passed using the `jphi-linterp` profile type.

In [ ]:
def solve_at_pressure(fraction):
    pressure = fraction*p_thermal
    pprime = np.gradient(pressure, psi_prof, edge_order=2)/(mygs.psi_bounds[1]-mygs.psi_bounds[0])

    mygs.set_targets(Ip=abs(eqdsk['ip']), pax=pressure[0])
    mygs.set_profiles(
        pp_prof={'type': 'linterp', 'x': psi_prof, 'y': pprime},
        ffp_prof={'type': 'jphi-linterp', 'x': psi_prof, 'y': jphi_baseline},
    )
    mygs.solve()
    return mygs.get_stats()['beta_n']

## Running GPEC

GPEC reads its settings from a `gpec.toml` file in a run directory, and writes its results to `gpec.h5` in the same directory. Here we analyze $n=1$ modes without a wall. To include a wall, change `wall` to, for example, `{'shape': 'conformal', 'a': 0.2}` for a wall that follows the plasma boundary at a distance of 0.2 times the minor radius. See the GPEC documentation for a description of all available settings.

In [ ]:
n_tor = 1
target_m, target_n = 2, 1
wall = {'shape': 'nowall'}

def write_gpec_toml(run_dir, eq_filename):
    wall_settings = '\n'.join('{0} = {1}'.format(key, json.dumps(value)) for key, value in wall.items())
    with open(os.path.join(run_dir, 'gpec.toml'), 'w') as fid:
        fid.write(f'''
[Equilibrium]
eq_filename = "{eq_filename}"
eq_type     = "efit"
jac_type    = "hamada"
grid_type   = "ldp"
psilow      = 1e-4
psihigh     = 0.995
mpsi        = 257
mtheta      = 514

[Wall]
{wall_settings}

[ForceFreeStates]
local_stability_flag = true
vac_flag = true
set_psilim_via_dmlim = true
dmlim = 0.2
qhigh = 1e3
qlow  = 1.02
nn_low  = {n_tor}
nn_high = {n_tor}
delta_mlow  = 8
delta_mhigh = 8
mthvac      = 960
HDF5_filename = "gpec.h5"
''')

Along with the ideal energy $\delta W$, GPEC computes the tearing stability index $\Delta'$ at each rational surface. A tearing mode is classically unstable when $\Delta' > 0$. However, in hot plasmas with good average curvature, resistive effects in the thin layer around the rational surface (the Glasser effect) raise this threshold to $\Delta' > \Delta_{crit}$, so we use $\Delta_{eff} = \Delta' - \Delta_{crit}$ as our tearing stability metric. GPEC calculates $\Delta_{crit}$ with its SLAYER (SLab inner LAYER) model, which needs the kinetic profiles as input.

The Julia function below, defined from Python with `jl.seval()`, runs GPEC on a run directory and then computes $\Delta_{crit}$ at the target surface from the same GPEC result.

In [ ]:
jl.seval('''
function run_gpec(path, m, n, psi, n_e, T_e, T_i, omega; zeff=1.5, chi_perp=1.0)
    gpec = GeneralizedPerturbedEquilibrium
    result = gpec.main([path])

    surfaces = filter(s -> m in s.m && n in s.n, result.ffs.surfaces)
    isempty(surfaces) && error("No $m/$n rational surface found")

    # The diamagnetic frequencies do not affect Delta_crit, they only need to differ to pass SLAYER's input checks
    profiles = gpec.Utilities.KineticProfiles(; psi, n_e, T_e, T_i, omega,
                                              omega_e=omega, omega_i=-omega .- 1.0)
    params = gpec.InnerLayer.SLAYER.build_slayer_inputs(result.ffs.equil, surfaces, profiles;
        mu_i=2.0, zeff=zeff, chi_perp=chi_perp, chi_tor=chi_perp, dc_type=:rfitzp)
    return params[1].dc_tmp
end
''')

Finally, we combine everything into a single function that, for a given pressure `fraction`:

1) Solves the equilibrium with TokaMaker and saves it as a gEQDSK file in its own run directory (`gpec_runs/fraction_X.XX`).
2) Writes the GPEC settings and runs GPEC, including $\Delta_{crit}$. The density is scaled together with the pressure.
3) Reads $\delta W$ and $\Delta'$ at the target surface from GPEC's `gpec.h5` output file.

In [ ]:
def evaluate_point(fraction):
    run_dir = os.path.join('gpec_runs', 'fraction_{0:.2f}'.format(fraction))
    os.makedirs(run_dir, exist_ok=True)

    beta_n = solve_at_pressure(fraction)
    mygs.save_eqdsk(os.path.join(run_dir, 'tokamaker.geqdsk'), nr=257, nz=257, truncate_eq=False, lcfs_pad=0.001)
    write_gpec_toml(run_dir, 'tokamaker.geqdsk')

    delta_crit = jl.run_gpec(run_dir, target_m, target_n, psi_prof,
                             fraction*kinetic['ne(m^-3)'].to_numpy(), kinetic['te(eV)'].to_numpy(),
                             kinetic['ti(eV)'].to_numpy(), kinetic['wexb(rad/s)'].to_numpy())

    with h5py.File(os.path.join(run_dir, 'gpec.h5'), 'r') as h5:
        dW = h5['ForceFreeStates/FreeBoundaryStability/eigenmode_energies'][0].real
        rational_m = h5['SingularSurfaces/rational_m'][()].flatten()
        rational_n = h5['SingularSurfaces/rational_n'][()].flatten()
        isurf = np.where((rational_m == target_m) & (rational_n == target_n))[0][0]
        delta_prime = h5['SingularSurfaces/Delta_prime_matrix'][isurf, isurf].real

    return {'fraction': fraction, 'beta_n': beta_n, 'dW': dW,
            'delta_prime': delta_prime, 'delta_crit': delta_crit}

## Run the scan

We now loop over a range of pressure fractions. The first point takes a few minutes while Julia compiles GPEC, and each point after that takes less than a minute. If a point fails, for example because TokaMaker does not converge, it is skipped and the scan continues.

In [ ]:
fractions = np.linspace(1.0, 1.8, 8)

scan_results = []
for fraction in fractions:
    try:
        scan_results.append(evaluate_point(fraction))
    except Exception as exc:
        print('Skipping fraction = {0:.2f}: {1}'.format(fraction, exc))

results = pd.DataFrame(scan_results)
results['delta_eff'] = results['delta_prime'] - results['delta_crit']
results

### Ideal kink stability

The plasma is ideally stable to the $n=1$ external kink while $\delta W > 0$. We find the ideal $\beta_N$ limit by interpolating to the point where $\delta W$ changes sign.

In [ ]:
results = results.sort_values('beta_n').reset_index(drop=True)
beta_n, dW = results['beta_n'].to_numpy(), results['dW'].to_numpy()

beta_limit = None
sign_change = np.where(np.diff(np.sign(dW)) != 0)[0]
if sign_change.size > 0:
    i = sign_change[0]
    beta_limit = beta_n[i] - dW[i]*(beta_n[i+1]-beta_n[i])/(dW[i+1]-dW[i])
    print('Ideal no-wall limit: beta_N = {0:.2f}'.format(beta_limit))

fig, ax = plt.subplots(1,1)
ax.plot(beta_n, dW, 'o-')
ax.axhline(0.0, color='gray', linewidth=0.8)
if beta_limit is not None:
    ax.axvline(beta_limit, color='tab:red', linestyle='--', linewidth=1.0, label=r'$\beta_N^{lim}$')
    ax.legend()
ax.set_xlabel(r'$\beta_N$ [%]')
_ = ax.set_ylabel(r'$\delta W$')

### Tearing stability

Finally, we plot $\Delta'$ and $\Delta_{eff}$ at the 2/1 surface. Since the tearing metrics are expected to behave poorly close to the ideal limit, we plot them against the normalized pressure $\beta/\beta_{lim}$ and only show the points below the limit. $I_p$, $B_0$ and the plasma shape are fixed in the scan, so $\beta/\beta_{lim}$ is equal to $\beta_N/\beta_N^{lim}$.

In [ ]:
if beta_limit is None:
    raise RuntimeError('No ideal beta limit found in the scan, try extending the range of fractions.')
below_limit = results[results['beta_n'] <= beta_limit]
beta_frac = below_limit['beta_n']/beta_limit

fig, ax = plt.subplots(1,1)
ax.plot(beta_frac, below_limit['delta_prime'], 'o--', color='tab:blue', label=r"$\Delta'$")
ax.plot(beta_frac, below_limit['delta_eff'], 'o-', color='tab:blue', label=r"$\Delta_{eff} = \Delta' - \Delta_{crit}$")
ax.axhline(0.0, color='gray', linewidth=0.8)
ax.set_xlim(beta_frac.min(), 1.0)
ax.set_xlabel(r'$\beta/\beta_{lim}$')
ax.set_ylabel(r"$\Delta'$ ({0}/{1} surface)".format(target_m, target_n))
_ = ax.legend()